Instalación de Dependencias

In [1]:
!pip -q install "SQLAlchemy>=2,<3" pydantic

Importación de Tipos de Datos Opcionales, Componentes Core de SQLAlchemy Y Herramientas ORM Modernas

In [2]:
from typing import Optional
from sqlalchemy import String, Float, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

Creación del Motor de Base de Datos (Engine)

In [3]:
engine = create_engine("sqlite:///clase_orm.db")

print("Engine creado:", engine)

Engine creado: Engine(sqlite:///clase_orm.db)


Declaración de la Clase Base Declarativa del ORM

In [4]:
class Base(DeclarativeBase):
    pass


In [5]:
Base

__main__.Base

Crear un modelo orm para representar clientes y compras de una tienda

In [6]:
from typing import List
from datetime import datetime
from sqlalchemy import String, Float, ForeignKey, DateTime
from sqlalchemy.orm import Mapped, mapped_column, relationship

# Limpiamos los metadatos registrados previamente para evitar conflictos al reejecutar la celda
Base.metadata.clear()

# --- 1. MODELO DE CLIENTES (USERS) ---
class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    email: Mapped[str] = mapped_column(String(100))
    average_spend: Mapped[float] = mapped_column(Float, default=0.0)

    # Relación uno a muchos (Un usuario tiene muchas compras)
    purchases: Mapped[List["Purchase"]] = relationship(back_populates="user", cascade="all, delete-orphan")

    def __repr__(self):
        return f"User(id={self.id}, name={self.name!r}, spend={self.average_spend})"

# --- 2. MODELO DE COMPRAS (PURCHASES) ---
class Purchase(Base):
    __tablename__ = "purchases"

    id: Mapped[int] = mapped_column(primary_key=True)
    amount: Mapped[float] = mapped_column(Float) # Monto de la compra
    date: Mapped[datetime] = mapped_column(DateTime, default=datetime.utcnow) # Fecha de la compra

    # Llave foránea que referencia al id de la tabla users
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))

    # Relación de vuelta hacia el usuario
    user: Mapped["User"] = relationship(back_populates="purchases")

    def __repr__(self):
        return f"Purchase(id={self.id}, amount={self.amount}, user_id={self.user_id})"

# --- 3. CREACIÓN DE LAS TABLAS EN LA BASE DE DATOS ---
Base.metadata.create_all(engine)

print("¡Tablas 'users' y 'purchases' creadas exitosamente!")

¡Tablas 'users' y 'purchases' creadas exitosamente!


In [7]:
from datetime import datetime
from sqlalchemy.orm import Session

# --- CREACIÓN E INSERCIÓN DE DATOS DE EJEMPLO ---
with Session(engine) as session:

    # 1. Instanciar los 10 usuarios
    user1 = User(name="Ana Pérez", email="ana.perez@email.com", average_spend=1500.0)
    user2 = User(name="Carlos Gómez", email="carlos.gomez@email.com", average_spend=3200.5)
    user3 = User(name="María Rodríguez", email="maria.rodriguez@email.com", average_spend=850.0)
    user4 = User(name="Juan López", email="juan.lopez@email.com", average_spend=4100.0)
    user5 = User(name="Sofía Martínez", email="sofia.martinez@email.com", average_spend=2300.0)
    user6 = User(name="Lucas Fernández", email="lucas.fernandez@email.com", average_spend=1250.75)
    user7 = User(name="Valentina Díaz", email="valentina.diaz@email.com", average_spend=5400.0)
    user8 = User(name="Mateo Ruiz", email="mateo.ruiz@email.com", average_spend=950.0)
    user9 = User(name="Lucía Torres", email="lucia.torres@email.com", average_spend=3100.25)
    user10 = User(name="Gabriel Sánchez", email="gabriel.sanchez@email.com", average_spend=1800.0)

    # Añadir y confirmar usuarios para que se generen sus IDs automáticos
    session.add_all([user1, user2, user3, user4, user5, user6, user7, user8, user9, user10])
    session.commit()

    # 2. Instanciar las 10 compras asociadas utilizando las relaciones ORM
    purchase1 = Purchase(amount=1500.0, user=user1)
    purchase2 = Purchase(amount=3200.5, user=user2)
    purchase3 = Purchase(amount=850.0, user=user3)
    purchase4 = Purchase(amount=4100.0, user=user4)
    purchase5 = Purchase(amount=2300.0, user=user5)
    purchase6 = Purchase(amount=500.0, user=user1)
    purchase7 = Purchase(amount=1200.0, user=user2)
    purchase8 = Purchase(amount=1250.75, user=user6)
    purchase9 = Purchase(amount=5400.0, user=user7)
    purchase10 = Purchase(amount=1800.0, user=user10)

    # Añadir y confirmar las compras
    session.add_all([
        purchase1, purchase2, purchase3, purchase4, purchase5,
        purchase6, purchase7, purchase8, purchase9, purchase10
    ])
    session.commit()

print("¡Los 10 usuarios y sus respectivas compras fueron insertados exitosamente en 'clase_orm.db'!")

¡Los 10 usuarios y sus respectivas compras fueron insertados exitosamente en 'clase_orm.db'!


Código de Consulta de Usuarios

In [8]:
from sqlalchemy import func

with Session(engine) as session:
    # 1. Identificar nombres duplicados en la base de datos
    stmt_dup_names = select(User.name).group_by(User.name).having(func.count(User.id) > 1)
    duplicate_names = session.scalars(stmt_dup_names).all()

    if duplicate_names:
        for name in duplicate_names:
            # Obtener registros ordenados por ID para este usuario
            stmt_users = select(User).where(User.name == name).order_by(User.id)
            records = session.scalars(stmt_users).all()

            # El primero se conserva, el resto se borra
            duplicates = records[1:]
            for dup in duplicates:
                session.delete(dup)
        session.commit()

    # 2. Realizar la consulta directa y limpia de los 10 usuarios finales
    stmt = select(User).order_by(User.id)
    users = session.scalars(stmt).all()

for user in users:
    print(user)

User(id=1, name='Ana Pérez', spend=1500.0)
User(id=2, name='Carlos Gómez', spend=3200.5)
User(id=3, name='María Rodríguez', spend=850.0)
User(id=4, name='Juan López', spend=4100.0)
User(id=5, name='Sofía Martínez', spend=2300.0)
User(id=6, name='Lucas Fernández', spend=1250.75)
User(id=7, name='Valentina Díaz', spend=5400.0)
User(id=8, name='Mateo Ruiz', spend=950.0)
User(id=9, name='Lucía Torres', spend=3100.25)
User(id=10, name='Gabriel Sánchez', spend=1800.0)


Código de Consulta de Compras con sus Relaciones

In [9]:
with Session(engine) as session:
    stmt_purchases = select(Purchase)
    purchases = session.scalars(stmt_purchases).all()

for purchase in purchases:
    print(f"Compra ID: {purchase.id} - Monto: ${purchase.amount} - ID de Usuario: {purchase.user_id}")

Compra ID: 1 - Monto: $1500.0 - ID de Usuario: 1
Compra ID: 2 - Monto: $3200.5 - ID de Usuario: 2
Compra ID: 3 - Monto: $850.0 - ID de Usuario: 3
Compra ID: 4 - Monto: $4100.0 - ID de Usuario: 4
Compra ID: 5 - Monto: $2300.0 - ID de Usuario: 5
Compra ID: 6 - Monto: $500.0 - ID de Usuario: 1
Compra ID: 7 - Monto: $1200.0 - ID de Usuario: 2
Compra ID: 8 - Monto: $1250.75 - ID de Usuario: 6
Compra ID: 9 - Monto: $5400.0 - ID de Usuario: 7
Compra ID: 10 - Monto: $1800.0 - ID de Usuario: 10


Filtrar Clientes por Gasto Promedio (User)

In [10]:
from sqlalchemy.orm import Session
from sqlalchemy import select

with Session(engine) as session:
    # Filtramos usuarios con un gasto promedio mayor a 2000.0
    stmt = select(User).where(User.average_spend > 2000.0)
    users = session.scalars(stmt).all()

print("--- Clientes con gasto promedio > 2000.0 ---")
for user in users:
    print(user)

--- Clientes con gasto promedio > 2000.0 ---
User(id=2, name='Carlos Gómez', spend=3200.5)
User(id=4, name='Juan López', spend=4100.0)
User(id=5, name='Sofía Martínez', spend=2300.0)
User(id=7, name='Valentina Díaz', spend=5400.0)
User(id=9, name='Lucía Torres', spend=3100.25)


Filtrar Compras por Monto Mínimo (Purchase)
Para la tabla de compras, puedes aplicar el mismo principio filtrando por el campo amount (monto). Por ejemplo, para buscar todas las compras que superen los $1500.0:

In [11]:
with Session(engine) as session:
    # Filtramos compras con un monto mayor a 1500.0
    stmt_purchases = select(Purchase).where(Purchase.amount > 1500.0)
    purchases = session.scalars(stmt_purchases).all()

print("\n--- Compras con un monto mayor a $1500.0 ---")
for purchase in purchases:
    print(f"Compra ID: {purchase.id} - Monto: ${purchase.amount} - ID de Usuario: {purchase.user_id}")


--- Compras con un monto mayor a $1500.0 ---
Compra ID: 2 - Monto: $3200.5 - ID de Usuario: 2
Compra ID: 4 - Monto: $4100.0 - ID de Usuario: 4
Compra ID: 5 - Monto: $2300.0 - ID de Usuario: 5
Compra ID: 9 - Monto: $5400.0 - ID de Usuario: 7
Compra ID: 10 - Monto: $1800.0 - ID de Usuario: 10


Filtro Avanzado: Consultar Compras de un Cliente Específico (Join con Relación)
Gracias a las relaciones ORM que configuramos previamente, puedes filtrar las compras de un usuario utilizando su nombre o sus atributos relacionados de forma directa:

In [12]:
with Session(engine) as session:
    # Buscamos las compras realizadas por usuarios cuyo nombre contenga 'Ana'
    stmt_join = select(Purchase).join(Purchase.user).where(User.name.like("%Ana%"))
    purchases_ana = session.scalars(stmt_join).all()

print("\n--- Compras realizadas por la clienta Ana ---")
for purchase in purchases_ana:
    print(f"Compra ID: {purchase.id} - Monto: ${purchase.amount} (Cliente ID: {purchase.user_id})")


--- Compras realizadas por la clienta Ana ---
Compra ID: 1 - Monto: $1500.0 (Cliente ID: 1)
Compra ID: 6 - Monto: $500.0 (Cliente ID: 1)


Modificar un objeto: tabla de Clientes (User)
Actualicemos el gasto promedio del cliente Carlos Gómez:

In [13]:
with Session(engine) as session:
    # Usamos .first() en lugar de .one() para evitar errores si hay nombres duplicados en la base de datos
    user = session.scalars(
        select(User).where(User.name == "Carlos Gómez")
    ).first()

    if user:
        # Modificamos el atributo deseado
        user.average_spend = 3500.0
        session.commit()
        print(f"Gasto promedio del cliente '{user.name}' (ID: {user.id}) actualizado exitosamente a ${user.average_spend}.")
    else:
        print("No se encontró ningún cliente con ese nombre.")

Gasto promedio del cliente 'Carlos Gómez' (ID: 2) actualizado exitosamente a $3500.0.
